# ASTraM Event Forecasting - Data Cleaning & EDA

This notebook performs the initial data ingestion, cleaning, and exploratory data analysis for the ASTraM event dataset.

In [15]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import geopandas as gpd
import requests

# Ensure data directory exists for outputs
os.makedirs('../data', exist_ok=True)

## 1. Data Ingestion

In [16]:
raw_file_path = r'../Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
df = pd.read_csv(raw_file_path, on_bad_lines='skip')
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (8170, 46)


## 2. Drop Dead Columns
As identified in the audit, these columns are entirely missing or too sparse to be useful (e.g., resource allocations).

In [17]:
dead_columns = ['meta_data', 'comment', 'map_file', 'assigned_to_police_id']
df = df.drop(columns=[c for c in dead_columns if c in df.columns])
print(f'Dataset shape after dropping dead columns: {df.shape}')

# Drop non-operational test/demo entries — system-test rows logged by operators
# to verify ASTraM was working, not real incidents. Confirmed: only 2 rows.
NON_OPERATIONAL_CAUSES = {'test_demo'}
before = len(df)
df = df[~df['event_cause'].isin(NON_OPERATIONAL_CAUSES)].copy()
print(f'Dropped {before - len(df)} non-operational rows (test_demo). New shape: {df.shape}')

Dataset shape after dropping dead columns: (8170, 42)
Dropped 0 non-operational rows (test_demo). New shape: (8170, 42)


## 3. Timezone Conversion (UTC to IST)
The timestamps are currently in UTC. We need them in IST (+05:30) to properly extract hour of day and day of week features later.

In [18]:
dt_cols = ['start_datetime', 'end_datetime', 'closed_datetime']

for col in dt_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    # Convert to IST
    df[col] = df[col].dt.tz_convert('Asia/Kolkata') if df[col].dt.tz else df[col].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')

## 4. Target Construction & Cleaning
We need to combine `end_datetime` (used for planned events) and `closed_datetime` (used for unplanned events) to get the true resolution time.

In [19]:
df['resolution_dt'] = df['end_datetime'].fillna(df['closed_datetime'])

# Compute duration in minutes
df['duration_minutes'] = (df['resolution_dt'] - df['start_datetime']).dt.total_seconds() / 60.0

# 1. Drop active incidents (censored data)
df = df[df['status'] != 'active'].copy()

# 2. Drop negative durations (data entry errors)
df = df[(df['duration_minutes'] >= 0) | df['duration_minutes'].isna()].copy()

# 3. Winsorize extreme outliers (cap at 48 hours / 2880 minutes)
MAX_DURATION = 48 * 60
df['duration_minutes'] = df['duration_minutes'].clip(upper=MAX_DURATION)

print(f"Final shape after basic cleaning: {df.shape}")
print(f"Rows with valid duration labels: {df['duration_minutes'].notna().sum()}")

Final shape after basic cleaning: (7114, 44)
Rows with valid duration labels: 3426


## 5. Exploratory Data Analysis

In [20]:
# Top event causes
cause_counts = df['event_cause'].value_counts().reset_index()
cause_counts.columns = ['event_cause', 'count']
fig = px.bar(cause_counts, x='event_cause', y='count', title='Incident Counts by Cause')
fig.show()

# Priority vs Closure Probability
closure_prob = df.groupby('priority')['requires_road_closure'].mean().reset_index()
fig2 = px.bar(closure_prob, x='priority', y='requires_road_closure', title='Probability of Road Closure by Priority')
fig2.show()

# Median Duration by Cause
median_duration = df.dropna(subset=['duration_minutes']).groupby('event_cause')['duration_minutes'].median().sort_values().reset_index()
fig3 = px.bar(median_duration, x='event_cause', y='duration_minutes', title='Median Duration (Minutes) by Cause')
fig3.show()

## 6. OSM Enrichment
We perform a spatial nearest-neighbor join to attach OpenStreetMap road classifications.

In [ ]:
roads = gpd.read_file('../data/export.geojson')
vehicle_classes = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "living_street",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
]
roads = roads[roads["highway"].isin(vehicle_classes)].copy()
roads_m = roads.to_crs(epsg=32643)

incidents = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
).to_crs(epsg=32643)

enriched = gpd.sjoin_nearest(
    incidents,
    roads_m[["highway", "lanes", "geometry"]],
    how="left",
    distance_col="dist_to_road_m",
)
enriched = enriched[~enriched.index.duplicated(keep='first')]

df["osm_highway_class"] = enriched["highway"]
df["osm_lanes"] = pd.to_numeric(enriched["lanes"], errors='coerce')
df["dist_to_nearest_road_m"] = enriched["dist_to_road_m"]
print("OSM Enrichment Complete!")

OSM Enrichment Complete!


## 7. Weather Enrichment
We fetch hourly precipitation from Open-Meteo Historical API and merge it into the dataset.

In [22]:
min_date = df['start_datetime'].min().tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
max_date = df['start_datetime'].max().tz_convert('Asia/Kolkata').strftime('%Y-%m-%d')
print(f"Fetching weather from {min_date} to {max_date}...")

url = f"https://archive-api.open-meteo.com/v1/archive?latitude=12.9716&longitude=77.5946&start_date={min_date}&end_date={max_date}&hourly=precipitation&timezone=Asia%2FKolkata"
response = requests.get(url)
response.raise_for_status()
data = response.json()

weather_df = pd.DataFrame({
    'time': pd.to_datetime(data['hourly']['time']),
    'precipitation_mm': data['hourly']['precipitation']
})
weather_df['time'] = weather_df['time'].dt.tz_localize('Asia/Kolkata').dt.tz_convert('UTC')

df['merge_hour'] = df['start_datetime'].dt.round('h')
weather_df['merge_hour'] = weather_df['time'].dt.round('h')
weather_df = weather_df.groupby('merge_hour', as_index=False)['precipitation_mm'].mean()

merged_df = pd.merge(df, weather_df[['merge_hour', 'precipitation_mm']], on='merge_hour', how='left')
merged_df.drop(columns=['merge_hour'], inplace=True)
merged_df['precipitation_mm'] = merged_df['precipitation_mm'].fillna(0.0)

df = merged_df
print("Weather Enrichment Complete!")

Fetching weather from 2023-11-10 to 2024-04-08...
Weather Enrichment Complete!


## 8. Export Final Augmented Data

In [23]:
df.to_csv('../data/augmented_astram_events.csv', index=False)
print("Saved augmented dataset to data/augmented_astram_events.csv")

Saved augmented dataset to data/augmented_astram_events.csv
